# Chapter 07: Instruction data, masking, and parameter-efficient adaptation


Instruction fine-tuning turns structured records into token sequences,
chooses which tokens contribute to loss, and balances model adaptation
against memory cost. The final exercise introduces LoRA as a low-rank
update to a frozen linear layer.


    This notebook is an independently written practice companion. It
    does not reproduce the book's prose, figures, data, or solutions.
    Read the [Chinese chapter](https://github.com/skindhu/Build-A-Large-Language-Model-CN/blob/main/cn-Book/7.%E6%8C%87%E4%BB%A4%E9%81%B5%E5%BE%AA%E5%BE%AE%E8%B0%83.md) for the complete narrative
    and exercise statements, and use the
    [canonical MIT companion code](https://github.com/rasbt/LLMs-from-scratch)
    as an additional reference.

    **Prerequisites:** Python 3.10+ and PyTorch 2.2+. Every example is
    synthetic, deterministic, CPU-friendly, and makes no network call.

    **Workflow**

    1. Read the concept and trace the shapes by hand.
    2. Replace one `LLM_TODO` and its `NotImplementedError`.
    3. Run that exercise's check cell.
    4. Explain the result before moving to the next exercise.
    5. Compare with `solution.py` only after making a serious attempt.

In [ ]:
from __future__ import annotations

import math
from collections.abc import Mapping
from dataclasses import dataclass
from typing import Literal

import torch
from torch import Tensor, nn


@dataclass(frozen=True)
class MemoryPlan:
    num_examples: int
    sequence_length: int
    micro_batch_size: int
    gradient_accumulation_steps: int
    tokens_per_update: int
    estimated_activation_mib: float

## Exercise 7.1: Format instruction records

    **Objective.** Render one record in two original prompt layouts while preserving field roles.


A prompt template serializes structured fields into plain text. The
exact delimiters are model conventions; the important invariant is that
instruction, optional context, and expected response remain
distinguishable and appear in a consistent order.

This notebook uses independently worded `alpaca_like` and `phi_like`
layouts for mechanics only. They are not copies of upstream templates.
Formatting should be deterministic so token counts and masks can be
reproduced.


    ### Data-flow walkthrough


The record contains an assignment, optional details, and a response.
The first style uses Markdown-like section labels; the second uses
user/assistant role tokens. Empty details receive an explicit placeholder
so record structure remains visible.


    ### Hints and common mistakes


Strip surrounding whitespace but do not discard meaningful content.
Reject an empty instruction and an unknown style. Keep the response in
the returned training string; inference formatting would omit it.

In [ ]:
def format_instruction(
    record: Mapping[str, str],
    style: Literal["alpaca_like", "phi_like"],
) -> str:
    # LLM_TODO_CH07_01_FORMAT
    raise NotImplementedError("render two original prompt layouts")

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
record = {
    "instruction": "Sort the symbols by code point.",
    "input": "z, a, m",
    "output": "a, m, z",
}
style_a = format_instruction(record, "alpaca_like")
style_b = format_instruction(record, "phi_like")
assert record["instruction"] in style_a and record["output"] in style_a
assert "<|user|>" in style_b and "<|assistant|>" in style_b
assert style_a != style_b
print("Exercise 7.1 passed")

## Exercise 7.2: Mask prompt labels

    **Objective.** Exclude a prompt prefix from next-token loss with an ignore index.


Language-model inputs and labels are often copies shifted by one token.
For response-focused instruction tuning, prompt-token label positions can
be replaced with an `ignore_index` such as `-100`. PyTorch cross entropy
omits those positions from its reduction.

Masking labels does not remove prompt tokens from the input. The model
still reads the instruction as context; it simply receives no direct
loss for reproducing that prefix.


    ### Data-flow walkthrough


For IDs `[4,5,6,7,8]` and prompt length three, cloned labels become
`[-100,-100,-100,7,8]`. The original input tensor must remain unchanged.


    ### Hints and common mistakes


Clone before assignment. Validate that prompt length is between zero and
sequence length. In a shifted-label pipeline, carefully define whether
the boundary refers to input or target positions.

In [ ]:
def mask_prompt_labels(
    input_ids: Tensor,
    prompt_length: int,
    ignore_index: int = -100,
) -> Tensor:
    # LLM_TODO_CH07_02_MASK
    raise NotImplementedError("clone labels and mask the prompt prefix")

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
ids = torch.tensor([4, 5, 6, 7, 8])
labels = mask_prompt_labels(ids, prompt_length=3)
assert labels.tolist() == [-100, -100, -100, 7, 8]
assert ids.tolist() == [4, 5, 6, 7, 8]
print("Exercise 7.2 passed")

## Exercise 7.3: Build a bounded memory plan

    **Objective.** Derive a toy microbatch and accumulation plan without downloading a dataset.


Dataset size, sequence length, hidden width, and batch size affect
memory differently. This exercise uses an explicitly labeled heuristic:
FP32 values times eight retained activation groups. It is not a hardware
predictor, but it demonstrates how to turn a budget into a bounded
microbatch.

Gradient accumulation separates microbatch memory from effective update
size. Several small forward/backward passes can contribute gradients
before one optimizer step. The token count per update is
`micro_batch * accumulation * sequence_length`.


    ### Data-flow walkthrough


Clamp average tokens to `max_sequence_length`, estimate bytes per
example, choose the largest microbatch under the activation budget, and
select enough accumulation steps to approach a small target effective
batch. Report the estimate transparently in MiB.


    ### Hints and common mistakes


Reject a budget that cannot fit one estimated example. Do not claim the
estimate includes parameters, optimizer state, allocator fragmentation,
or attention's implementation-specific temporaries.

In [ ]:
def build_memory_plan(
    num_examples: int,
    average_tokens: int,
    max_sequence_length: int,
    hidden_dim: int,
    activation_budget_mib: float,
) -> MemoryPlan:
    # LLM_TODO_CH07_03_PLAN
    raise NotImplementedError(
        "derive bounded microbatch and accumulation values"
    )

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
plan = build_memory_plan(
    10_000, 128, 256, 256, activation_budget_mib=8
)
assert plan.sequence_length == 128
assert 0 < plan.estimated_activation_mib <= 8
assert plan.micro_batch_size >= 1
assert plan.gradient_accumulation_steps >= 1
print("Exercise 7.3 passed")

## Exercise 7.4: Add a LoRA update

    **Objective.** Freeze a base linear layer and learn a scaled low-rank residual.


A linear layer computes $xW^T+b$. LoRA freezes $W$ and adds
$\Delta W=BA$, where $A\in\mathbb{R}^{r\times D_{in}}$ and
$B\in\mathbb{R}^{D_{out}\times r}$. The forward pass becomes
$xW^T+b+(\alpha/r)xA^TB^T$.

If $r$ is much smaller than both feature dimensions, trainable values
drop from roughly $D_{in}D_{out}$ to
$r(D_{in}+D_{out})$. Initializing $B$ to zero makes the initial update
zero, so wrapping a layer does not immediately change its output.


    ### Data-flow walkthrough


Wrap a `16 -> 12` linear layer with rank two. Trainable low-rank factors
contain `2*(16+12)=56` values, fewer than the base layer's trainable
weight and bias. Before training, wrapped and base outputs should match.


    ### Hints and common mistakes


Freeze both base weight and bias. Register factors as `nn.Parameter`.
Validate rank bounds and apply transposes according to PyTorch's linear
convention. Zero-initialize only one factor; zeroing both can delay
useful gradients.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(
        self, base: nn.Linear, rank: int, alpha: float = 1.0
    ) -> None:
        super().__init__()
        # LLM_TODO_CH07_04_LORA
        raise NotImplementedError(
            "freeze base and create trainable low-rank factors"
        )

    def forward(self, inputs: Tensor) -> Tensor:
        raise NotImplementedError("add the scaled low-rank update")

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
torch.manual_seed(9)
base = nn.Linear(16, 12)
sample = torch.randn(3, 16)
expected = base(sample).detach()
base_count = sum(p.numel() for p in base.parameters())
lora = LoRALinear(base, rank=2, alpha=4)
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
assert torch.allclose(lora(sample), expected)
assert trainable == 2 * (16 + 12) < base_count
print("Exercise 7.4 passed")

## Chapter 07 synthesis


- Which prompt fields must be present during training versus inference?
- Why are prompt tokens retained as inputs when their labels are masked?
- What does the memory heuristic omit, and how would you measure reality?
- Why does zero-initializing one LoRA factor preserve the base output?


    After answering these questions, run the chapter's `solution.py`
    from a terminal to compare behavior. The reference file is compact
    by design; the reasoning in this notebook is the main lesson.